# Duplicate Analysis: `suggest_impression_sanitized_v3`

This notebook investigates duplicate rows in `search_terms_derived.suggest_impression_sanitized_v3`. The v3 query joins impression pings from two sources (`quicksuggest_impression_v1` and `quick_suggest_v1`) with `merino_log_sanitized_v3` to enrich them with query-text data. Duplicates could originate from either the JOIN or the UNION ALL

In [ ]:
%load_ext bigquery_magics

## Step 1: Measure the duplicate rate

Count how many `request_id`s appear more than once in `suggest_impression_sanitized_v3` for yesterday. `excess_rows` is the number of extra copies beyond one row per request, and `dupe_rate` is that count as a fraction of total rows.

In [14]:
%%bigquery --project mozdata

WITH base AS (
  SELECT
    DATE(submission_timestamp) AS submission_date,
    request_id,
    COUNT(*) AS row_count
  FROM
    `moz-fx-data-shared-prod.search_terms_derived.suggest_impression_sanitized_v3`
  WHERE
    DATE(submission_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  GROUP BY
    1,
    2
),
summary AS (
  SELECT
    submission_date,
    SUM(row_count) AS total_rows,
    COUNT(DISTINCT request_id) AS distinct_request_ids,
    SUM(row_count) - COUNT(DISTINCT request_id) AS excess_rows,
    COUNTIF(row_count > 1) AS duplicated_request_ids,
    SAFE_DIVIDE(
      SUM(row_count) - COUNT(DISTINCT request_id),
      SUM(row_count)
    ) AS dupe_rate
  FROM
    base
  GROUP BY
    1
)
SELECT
  *
FROM
  summary

/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/job/query.py:2083: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  
/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2660: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  ("start", pyarrow.timestamp("us")),


,submission_date,total_rows,distinct_request_ids,excess_rows,duplicated_request_ids,dupe_rate
0,2026-04-21,2552775,1195192,1357583,308740,0.531807


## Step 2: Identify the fan-out source

Two mechanisms could produce duplicate `request_id`s in the output:

1. **JOIN fan-out** - `merino_log_sanitized_v3` has multiple rows per `request_id`, so each impression row multiplies when joined.
2. **UNION ALL overlap** - the same `request_id` appears in both `quicksuggest_impression_v1` (legacy ping) and `quick_suggest_v1` (Glean ping), entering the union twice before the join.

This query measures how many request IDs are duplicated by each mechanism independently.

In [15]:
%%bigquery --project mozdata

-- Diagnose dupe source: JOIN fan-out vs UNION ALL overlap
WITH sanitized_fanout AS (
  SELECT
    request_id,
    COUNT(*) AS n
  FROM
    `moz-fx-data-shared-prod.search_terms_derived.merino_log_sanitized_v3`
  WHERE
    DATE(`timestamp`) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  GROUP BY
    1
  HAVING
    COUNT(*) > 1
),
union_overlap AS (
  SELECT
    request_id,
    COUNT(*) AS n
  FROM
    (
      SELECT
        request_id
      FROM
        `moz-fx-data-shared-prod.contextual_services_stable.quicksuggest_impression_v1`
      WHERE
        DATE(submission_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
      UNION ALL
      SELECT
        metrics.string.quick_suggest_request_id AS request_id
      FROM
        `moz-fx-data-shared-prod.firefox_desktop_stable.quick_suggest_v1`
      WHERE
        DATE(submission_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
        AND metrics.string.quick_suggest_ping_type = 'quicksuggest-impression'
    )
  GROUP BY
    1
  HAVING
    COUNT(*) > 1
)
SELECT
  'merino_log_sanitized_v3 fan-out (JOIN)' AS dupe_source,
  COUNT(*) AS duplicated_request_ids,
  SUM(n) AS total_rows_from_duped_ids,
  SUM(n) - COUNT(*) AS excess_rows
FROM
  sanitized_fanout
UNION ALL
SELECT
  'impression UNION ALL overlap' AS dupe_source,
  COUNT(*) AS duplicated_request_ids,
  SUM(n) AS total_rows_from_duped_ids,
  SUM(n) - COUNT(*) AS excess_rows
FROM
  union_overlap

/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/job/query.py:2083: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  
/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2660: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  ("start", pyarrow.timestamp("us")),


,dupe_source,duplicated_request_ids,total_rows_from_duped_ids,excess_rows
0,impression UNION ALL overlap,13,946082,946069
1,merino_log_sanitized_v3 fan-out (JOIN),37210716,86757023,49546307


## Step 3: Understand the nature of the Merino fan-out

JOIN fan-out from `merino_log_sanitized_v3` is the dominant source by a wide margin. Now we check *why* that table has multiple rows per `request_id`. Are the copies meaningfully different (e.g., different query text, session, or timestamp) or are they exact duplicates?

For each column we count how many duplicated `request_id`s have any variation across their fan-out rows. A count of 0 for every column means all copies are identical.

In [16]:
%%bigquery --project mozdata

-- For each column in merino_log_sanitized_v3, count how many duplicate request_ids
-- have variation in that column. Tells us which columns differ across the fan-out rows.
WITH dupe_rows AS (
  SELECT *
  FROM `moz-fx-data-shared-prod.search_terms_derived.merino_log_sanitized_v3`
  WHERE
    DATE(`timestamp`) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
    AND request_id IN (
      SELECT request_id
      FROM `moz-fx-data-shared-prod.search_terms_derived.merino_log_sanitized_v3`
      WHERE DATE(`timestamp`) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
      GROUP BY 1
      HAVING COUNT(*) > 1
    )
),
per_request AS (
  SELECT
    request_id,
    COUNT(*)                             AS row_count,
    COUNT(DISTINCT query)         > 1    AS query_varies,
    COUNT(DISTINCT session_id)    > 1    AS session_id_varies,
    COUNT(DISTINCT sequence_no)   > 1    AS sequence_no_varies,
    COUNT(DISTINCT UNIX_SECONDS(`timestamp`)) > 1 AS timestamp_varies,
    COUNT(DISTINCT dma)           > 1    AS dma_varies,
    COUNT(DISTINCT form_factor)   > 1    AS form_factor_varies,
    COUNT(DISTINCT browser)       > 1    AS browser_varies,
    COUNT(DISTINCT os_family)     > 1    AS os_family_varies,
  FROM dupe_rows
  GROUP BY request_id
)
SELECT
  COUNT(*)                                  AS total_duped_request_ids,
  COUNTIF(query_varies)                     AS query_varies,
  COUNTIF(session_id_varies)                AS session_id_varies,
  COUNTIF(sequence_no_varies)               AS sequence_no_varies,
  COUNTIF(timestamp_varies)                 AS timestamp_varies,
  COUNTIF(dma_varies)                       AS dma_varies,
  COUNTIF(form_factor_varies)               AS form_factor_varies,
  COUNTIF(browser_varies)                   AS browser_varies,
  COUNTIF(os_family_varies)                 AS os_family_varies,
FROM per_request

/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/job/query.py:2083: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  
/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2660: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  ("start", pyarrow.timestamp("us")),


,total_duped_request_ids,query_varies,session_id_varies,sequence_no_varies,timestamp_varies,dma_varies,form_factor_varies,browser_varies,os_family_varies
0,37210716,0,0,0,0,0,0,0,0


## Step 4: Inspect sample duplicate rows

Every column variance count is 0 - the fan-out rows are exact duplicates, not meaningful variants. We pull 5 duplicated `request_id`s and display all their rows to confirm visually and look for any distinguishing signal (e.g., user-agent, source system).

## Step 4: Check for duplicates within each impression source table

The sample rows all show `browser = Go-http-client` - the HTTP client used by the Merino server itself, not a real Firefox user. This points to Merino logging the same request multiple times upstream.

As a separate check, we look for `request_id` duplicates within `quicksuggest_impression_v1` and `quick_suggest_v1` directly, independent of the Merino join, to see whether the impression sources also contribute their own duplicates.

In [20]:
%%bigquery --project mozdata

-- Check for duplicate request_ids within each impression source table independently
WITH quicksuggest_dupes AS (
  SELECT
    request_id,
    COUNT(*) AS n
  FROM
    `moz-fx-data-shared-prod.contextual_services_stable.quicksuggest_impression_v1`
  WHERE
    DATE(submission_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
  GROUP BY 1
  HAVING COUNT(*) > 1
),
glean_dupes AS (
  SELECT
    metrics.string.quick_suggest_request_id AS request_id,
    COUNT(*) AS n
  FROM
    `moz-fx-data-shared-prod.firefox_desktop_stable.quick_suggest_v1`
  WHERE
    DATE(submission_timestamp) = DATE_SUB(CURRENT_DATE(), INTERVAL 1 DAY)
    AND metrics.string.quick_suggest_ping_type = 'quicksuggest-impression'
  GROUP BY 1
  HAVING COUNT(*) > 1
)
SELECT
  'quicksuggest_impression_v1' AS source_table,
  COUNTIF(request_id IS NULL)     AS null_request_ids,
  COUNTIF(request_id IS NOT NULL) AS non_null_duplicated_request_ids,
  SUM(n) - COUNT(*)               AS excess_rows
FROM quicksuggest_dupes
UNION ALL
SELECT
  'quick_suggest_v1 (glean)' AS source_table,
  COUNTIF(request_id IS NULL)     AS null_request_ids,
  COUNTIF(request_id IS NOT NULL) AS non_null_duplicated_request_ids,
  SUM(n) - COUNT(*)               AS excess_rows
FROM glean_dupes

/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/job/query.py:2083: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  
/Users/chelseybeck/Documents/GitHub/bigquery-etl/venv/lib/python3.11/site-packages/google/cloud/bigquery/table.py:2660: UserWarning: A progress bar was requested, but there was an error loading the tqdm library. Please install tqdm to use the progress bar functionality.
  ("start", pyarrow.timestamp("us")),


,source_table,null_request_ids,non_null_duplicated_request_ids,excess_rows
0,quicksuggest_impression_v1,1,0,99123
1,quick_suggest_v1 (glean),1,8,846941


## Findings

| Source | Mechanism | Duplicated `request_id`s | Excess rows |
|---|---|---|---|
| `merino_log_sanitized_v3` | JOIN fan-out | ~37M | ~49.5M |
| `quick_suggest_v1` (Glean) | within-source dupes | 8 non-null | ~847K |
| `quicksuggest_impression_v1` | within-source dupes | 0 non-null | ~99K (null IDs only) |

**Root cause:** `merino_log_sanitized_v3` contains multiple identical rows per `request_id`. Because `suggest_impression_sanitized_v3` joins impressions to this table on `request_id`, every duplicate Merino row multiplies the impression row, producing the ~53% dupe rate seen in Step 1.

**Fix:** Deduplicate `merino_log_sanitized_v3` on `request_id` before joining, e.g. with a `QUALIFY ROW_NUMBER() OVER (PARTITION BY request_id ORDER BY timestamp) = 1` filter inside the CTE that reads from that table.